# Sistemas de 1ª ordem: simulação

Notebook da **Biblioteca Volt e Meia**. A teoria está na página
[Sistemas de 1ª ordem](https://arthurhac99.github.io/biblioteca-volt-e-meia/materias/controle/controle-de-sistemas-dinamicos/primeira-ordem/).

**Como usar:** clique em cada célula de código e aperte `Shift + Enter` para rodar, de cima para baixo.
Nada que você mudar aqui altera o original. Para guardar suas alterações, use *Arquivo → Salvar uma cópia no Drive*.

Função de transferência estudada:

$$G(s) = \frac{K}{\tau s + 1}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True

## 1. Resposta ao degrau

Para um degrau de amplitude $A$, a saída é $y(t) = K A \left(1 - e^{-t/\tau}\right)$.

Mude os valores de `K`, `tau` e `A` na célula abaixo e rode de novo para ver o efeito.

In [ ]:
K = 2.0     # ganho estático
tau = 1.5   # constante de tempo, em segundos
A = 5.0     # amplitude do degrau

t = np.linspace(0, 6 * tau, 600)
y = K * A * (1 - np.exp(-t / tau))
y_final = K * A

plt.plot(t, y, lw=2, label="saída y(t)")
plt.axhline(y_final, ls="--", c="gray", label=f"valor final = {y_final:.2f}")
plt.axhline(0.632 * y_final, ls=":", c="tab:red", label="63,2 % do final")
plt.axvline(tau, ls=":", c="tab:red")
plt.axvline(4 * tau, ls="-.", c="tab:green", label=f"acomodação (4τ) = {4 * tau:.2f} s")
plt.xlabel("tempo (s)")
plt.ylabel("y(t)")
plt.title("Resposta ao degrau de um sistema de 1ª ordem")
plt.legend()
plt.show()

print(f"Constante de tempo τ = {tau} s")
print(f"Tempo de subida (10 % a 90 %) ≈ 2,2τ = {2.2 * tau:.2f} s")
print(f"Tempo de acomodação (2 %) ≈ 4τ = {4 * tau:.2f} s")

## 2. Versão interativa

Arraste os controles e veja a curva mudar na hora. Repare:

- `K` muda **até onde** a curva vai, mas não muda **o tempo** para chegar lá.
- `τ` muda **a velocidade**, mas não muda o valor final.

In [ ]:
from ipywidgets import interact, FloatSlider

def desenhar(K=1.0, tau=1.0, A=1.0):
    t = np.linspace(0, 10, 600)
    y = K * A * (1 - np.exp(-t / tau))
    plt.plot(t, y, lw=2)
    plt.axhline(K * A, ls="--", c="gray")
    plt.axvline(tau, ls=":", c="tab:red", label=f"τ = {tau:.1f} s")
    plt.xlim(0, 10)
    plt.ylim(0, 11)
    plt.xlabel("tempo (s)")
    plt.ylabel("y(t)")
    plt.legend(loc="lower right")
    plt.show()

interact(desenhar,
         K=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="K"),
         tau=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="τ (s)"),
         A=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="A"));

## 3. Exemplo: carga de um capacitor (circuito RC)

Um circuito RC série é um sistema de 1ª ordem com $K = 1$ e $\tau = RC$.
Com $R = 10\ \text{k}\Omega$ e $C = 100\ \mu\text{F}$, esperamos $\tau = 1\ \text{s}$.

In [ ]:
R = 10e3      # ohms
C = 100e-6    # farads
Vin = 12.0    # volts

tau_rc = R * C
t = np.linspace(0, 5 * tau_rc, 500)
Vc = Vin * (1 - np.exp(-t / tau_rc))

plt.plot(t, Vc, lw=2, label="tensão no capacitor")
plt.axhline(Vin, ls="--", c="gray", label=f"fonte = {Vin} V")
for n in range(1, 5):
    v = Vin * (1 - np.exp(-n))
    plt.plot(n * tau_rc, v, "o", c="tab:red")
    plt.annotate(f"{n}τ: {100 * v / Vin:.1f} %".replace(".", ","), (n * tau_rc, v),
                 textcoords="offset points", xytext=(6, -14))
plt.xlabel("tempo (s)")
plt.ylabel("Vc (V)")
plt.title(f"Circuito RC: τ = RC = {tau_rc:.2f} s")
plt.legend(loc="lower right")
plt.show()

## 4. O mesmo resultado com a biblioteca de controle

Na prática, engenheiros de controle descrevem o sistema pela função de transferência e deixam a
biblioteca calcular a resposta. O `scipy.signal` faz isso: o numerador é `[K]` e o denominador é `[τ, 1]`,
que representa $\tau s + 1$.

In [ ]:
from scipy import signal

K, tau = 2.0, 1.5
G = signal.TransferFunction([K], [tau, 1])
t_sim, y_sim = signal.step(G, T=np.linspace(0, 6 * tau, 600))

y_formula = K * (1 - np.exp(-t_sim / tau))

plt.plot(t_sim, y_sim, lw=3, alpha=0.6, label="scipy.signal.step")
plt.plot(t_sim, y_formula, "k--", label="fórmula analítica")
plt.xlabel("tempo (s)")
plt.ylabel("y(t)")
plt.title("As duas curvas coincidem")
plt.legend()
plt.show()

print("Maior diferença entre as duas:", np.max(np.abs(y_sim - y_formula)))

## 5. Descobrindo τ a partir de dados de um experimento

No laboratório você não conhece $\tau$: aplica um degrau, mede a saída e precisa descobrir.
Abaixo geramos dados com ruído (como viriam de um sensor) e estimamos $\tau$ de dois jeitos:

1. **Método dos 63,2 %:** o instante em que a saída chega a 63,2 % do valor final.
2. **Ajuste de curva:** encontrar o $K$ e o $\tau$ que fazem a fórmula passar o mais perto possível dos pontos.

In [ ]:
from scipy.optimize import curve_fit

rng = np.random.default_rng(42)
K_real, tau_real = 3.0, 2.0          # valores "escondidos" do experimento
t_med = np.linspace(0, 12, 120)
y_med = K_real * (1 - np.exp(-t_med / tau_real)) + rng.normal(0, 0.06, t_med.size)

# Método 1: 63,2 % do valor final (média dos últimos pontos, já estabilizados)
y_final_est = y_med[-15:].mean()
indice = np.argmax(y_med >= 0.632 * y_final_est)
tau_632 = t_med[indice]

# Método 2: ajuste de curva por mínimos quadrados
def modelo(t, K, tau):
    return K * (1 - np.exp(-t / tau))

(K_fit, tau_fit), _ = curve_fit(modelo, t_med, y_med, p0=[1.0, 1.0])

plt.plot(t_med, y_med, ".", alpha=0.6, label="medições")
plt.plot(t_med, modelo(t_med, K_fit, tau_fit), "r", lw=2, label="curva ajustada")
plt.axhline(0.632 * y_final_est, ls=":", c="gray")
plt.axvline(tau_632, ls=":", c="gray", label=f"63,2 % em t = {tau_632:.2f} s")
plt.xlabel("tempo (s)")
plt.ylabel("y medido")
plt.legend(loc="lower right")
plt.show()

print(f"τ real (escondido):        {tau_real:.2f} s")
print(f"τ pelo método dos 63,2 %:  {tau_632:.2f} s")
print(f"τ pelo ajuste de curva:    {tau_fit:.2f} s   (K ajustado = {K_fit:.2f})")

## 6. Para praticar

1. Na seção 1, dobre o valor de `tau`. O que acontece com o tempo de acomodação? E com o valor final?
2. Na seção 3, troque o capacitor por um de 470 µF. Quanto tempo ele leva para carregar 98 %?
3. Na seção 5, aumente o ruído de `0.06` para `0.3`. Qual dos dois métodos fica mais confiável? Por quê?

---

Achou um erro ou tem uma ideia para este notebook? Conte pela página
[Como contribuir]({SITE}/contribuir/).